# Case Study 53 - Liver Cirrhosis & Disease Pattern Classification
**Course:** Machine Learning - B.Tech CSE (2024-28) - Semester V  
**Project Title:** *HepatoPattern - Classification of Liver Disease & Cirrhosis from Blood Tests*

---

Name: Rohan Vashisht

Cohort: Jensen Huang

Roll No.: 150096724132

---

Streamlit App: https://hepato-pattern-ml-rohanv.streamlit.app/

Google Colab:  https://colab.research.google.com/drive/1WG9aCzu0jXaWvWpKdhbmRw1qDQhmHv1C

UCI Dataset:   https://archive.ics.uci.edu/dataset/571/hcv+data

GitHub Repo:   https://github.com/RohanVashisht1234/hepato-pattern-ml

---

## 1. Problem Definition & Objectives

### 1.1 Objective
The goal is to develop a machine learning system that classifies whether a patient has **Active Liver Disease (Hepatitis / Fibrosis / Cirrhosis)** or has **Normal Liver Function (Healthy Blood Donor)** using standard blood test results.

* **Class 0 (Negative) - Healthy Blood Donors**: Certified healthy blood donors with normal liver enzyme levels (`0=Blood Donor` and `0s=suspect Blood Donor`).
* **Class 1 (Positive) - Active Liver Pathology**: Patients diagnosed with progressive liver disease (`1=Hepatitis`, `2=Fibrosis`, `3=Cirrhosis`).

### 1.2 Core Machine Learning Setup
* **Task Type**: Supervised Binary Classification.
* **Input Features**: 12 routine and biochemical blood markers (Age, Sex, ALT, AST, Bilirubin, Albumin, etc.).
* **Output**: Predicted class (0 = Healthy, 1 = Liver Disease) and disease probability.
* **Primary Metrics**: Accuracy, ROC-AUC, Recall, Precision, and Confusion Matrix.

## 2. Dataset Loading & Inspection

### 2.1 Clinical Dataset Origin (UCI Machine Learning Repository)
The dataset used in this project is an authentic clinical cohort obtained from the official **UCI Machine Learning Repository**:
* **Repository Source:** UCI Machine Learning Repository (Dataset ID #571)
* **Dataset Name:** HCV Data (Hepatitis C & Liver Disease Progression)
* **Direct Zip URL:** `https://archive.ics.uci.edu/static/public/571/hcv+data.zip`
* **Clinical Origin:** Hannover Medical School (Medizinische Hochschule Hannover, Germany)
* **Authors:** Dr. Ralf Lichtinghagen, Prof. Frank Klawonn, Gottfried Uhlig
* **Patient Records:** 615 real patient observations (540 healthy blood donor controls, 75 liver disease progression patients across Hepatitis, Fibrosis, and Cirrhosis)

The code block below automatically fetches the official ZIP archive directly from the UCI server if needed, extracts the original clinical file (`hcvdat0.csv`), and verifies the dataset integrity.

In [ ]:
!pip install pandas matplotlib seaborn scikit-learn joblib

In [ ]:
import os
import io
import urllib.request
import zipfile
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]

UCI_DATASET_DOWNLOAD_URL = "https://archive.ics.uci.edu/static/public/571/hcv+data.zip"
LOCAL_CSV_FILE_PATH = "cirrhosis.csv"


def download_uci_dataset_if_needed(dataset_download_url, local_csv_file_path):
    if os.path.exists(local_csv_file_path):
        print(f"Verified dataset already present locally: '{local_csv_file_path}'")
        print(f"Official UCI Repository Source: {dataset_download_url}")
        return

    print("Downloading dataset from UCI Machine Learning Repository:")
    print(f"URL: {dataset_download_url}")

    download_request = urllib.request.Request(
        dataset_download_url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(download_request) as download_response:
        downloaded_zip_data = io.BytesIO(download_response.read())

    with zipfile.ZipFile(downloaded_zip_data) as downloaded_zip_file:
        with downloaded_zip_file.open("hcvdat0.csv") as dataset_file:
            liver_disease_dataframe = pd.read_csv(dataset_file)

    liver_disease_dataframe.drop(
        columns=["Unnamed: 0"],
        errors="ignore"
    ).to_csv(local_csv_file_path, index=False)

    print(f"Successfully downloaded and extracted to '{local_csv_file_path}'.")


download_uci_dataset_if_needed(
    UCI_DATASET_DOWNLOAD_URL,
    LOCAL_CSV_FILE_PATH
)

liver_disease_dataframe = pd.read_csv(LOCAL_CSV_FILE_PATH)

liver_disease_dataframe.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
    inplace=True
)

print(
    f"Dataset Shape: {liver_disease_dataframe.shape[0]} rows, "
    f"{liver_disease_dataframe.shape[1]} columns"
)

liver_disease_dataframe.head()

Verified dataset already present locally: 'cirrhosis.csv'
Official UCI Repository Source: https://archive.ics.uci.edu/static/public/571/hcv+data.zip
Dataset Shape: 615 rows, 13 columns


,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT
0,0=Blood Donor,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0
1,0=Blood Donor,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5
2,0=Blood Donor,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3
3,0=Blood Donor,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7
4,0=Blood Donor,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7


In [ ]:
# Check missing values
missing_value_counts = liver_disease_dataframe.isnull().sum()

print("Missing Values Count:")
print(missing_value_counts[missing_value_counts > 0])

Missing Values Count:
ALB      1
ALP     18
ALT      1
CHOL    10
PROT     1
dtype: int64
